# **LLMs and Langchain Framework - Assignment**

**Question 1: What are Large Language Models (LLMs) and how do they function?**

Large Language Models (LLMs) are AI systems trained on massive datasets using neural networks and transformer architecture to understand, summarize, generate, and predict content.

They function by analyzing tokenized data to predict the next word in a sequence based on context. Common examples include OpenAI's GPT-4, Gemini, and Llama.

**Question 2: Discuss the impact of LLMs on traditional software development approaches.**

Large Language Models (LLMs) are transforming software development from manual coding to AI-assisted prompt engineering, significantly enhancing developer productivity by automating boilerplate generation, debugging, and testing. This shifts the focus toward code review, integration, and verification, accelerating development cycles while requiring new skills for AI collaboration. However, it raises risks regarding code reliability, over-reliance, and security, mandating a hybrid human-AI "Generator-Verifier" approach.

**Question 3: What are the key advantages and limitations of using LLMs in real-world applications?**

LLMs offer significant advantages in real-world applications by providing high-quality content generation, 24/7 customer support automation, language translation, and improved coding productivity. They excel at processing vast datasets to enhance operational efficiency.

However, key limitations include the propensity for hallucinations (fabricating information), potential bias in outputs, high computational costs, and a lack of true reasoning or contextual understanding.

**Question 4: Describe how different industries are being transformed by the use of LLMs. Provide examples.**

LLMs are revolutionising industries by automating complex cognitive tasks. In healthcare, they assist in diagnosing diseases and summarizing patient records. Finance uses them for fraud detection and automated reporting, while customer service is transformed through sophisticated AI chatbots that provide instant, human-like support, significantly boosting operational efficiency.

**Question 5: Compare and contrast Langchain and LamaIndex. What unique problems does each solve?**

LangChain is a general-purpose orchestration framework, while LlamaIndex is a specialized data-centric framework. LangChain focuses on modular components for creating agents and complex chains, whereas LlamaIndex specializes in data ingestion, indexing, and retrieval-augmented generation (RAG).

Both are often used together to build sophisticated, data-aware applications.

**Question 6: Implement a basic Langchain pipeline using OpenAI’s LLM to answer questions based on a user input prompt.**

In [ ]:
! pip install langchain langchain-openai python-dotenv

In [5]:
OPENAI_API_KEY="your_openai_api_key"

In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Load environment variables
load_dotenv()
# Ensure the API key is set
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable not set")

# 2. Initialize the OpenAI Chat Model
# Using a chat model is common for conversation-tuned models like gpt-3.5-turbo.
# Temperature controls creativity (0.0 is deterministic, higher is more creative).
llm = ChatOpenAI(model="gpt-3.5-turbo-instruct", temperature=0.7)

# 3. Define a Prompt Template
# This template provides instructions and a placeholder for the user's input.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions accurately and concisely."),
    ("human", "{question}")
])

# 4. Define an Output Parser
# The StrOutputParser ensures the output from the LLM is a clean string.
output_parser = StrOutputParser()

# 5. Build the Chain using LangChain Expression Language (LCEL)
# The pipe (|) operator chains the components together: Prompt | LLM | Output Parser.
chain = prompt_template | llm | output_parser

# 6. Invoke the Chain with a User Input
user_question = "What is the capital of France?"
response = chain.invoke({"question": user_question})

# 7. Print the Result
print(f"Question: {user_question}")
print(f"Answer: {response}")

**Question 7: Integrate Langchain with a third-party API (e.g., weather, news) and show how responses can be generated via LLMs.**

In [ ]:
!pip install langchain langchain-openai langchain-community requests python-dotenv

In [ ]:
OPENAI_API_KEY="your_openai_api_key"
OPENWEATHERMAP_API_KEY="your_openweathermap_api_key"

In [ ]:
import requests
import os
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv()
OPENWEATHERMAP_API_KEY = os.getenv("OPENWEATHERMAP_API_KEY")

@tool
def get_current_weather(city: str) -> str:
    """
    Fetches the current weather for a given city (e.g., "London,GB").
    The input must be a string representing the city name.
    """
    try:
        url = f"http://api.openweathermap.org{city}&appid={OPENWEATHERMAP_API_KEY}&units=metric"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        if data.get("cod") != 200:
            return f"Error: {data.get('message', 'City not found')}"

        weather_desc = data["weather"][0]["description"]
        temp = data["main"]["temp"]
        return f"The current weather in {city} is {temp}°C with {weather_desc}."
    except requests.exceptions.RequestException as e:
        return f"An error occurred: {e}"

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Initialize the agent with the tool, LLM, and agent type
agent = initialize_agent(
    tools=[get_current_weather],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [ ]:
# Example usage:
user_query_1 = "Hello! How are you today?"
print(f"User: {user_query_1}")
response_1 = agent.run(user_query_1)
print(f"AI: {response_1}\n")

user_query_2 = "What's the weather like in Tokyo today?"
print(f"User: {user_query_2}")
response_2 = agent.run(user_query_2)
print(f"AI: {response_2}\n")

**Question 8: Create a LamaIndex implementation that indexes a local text file and retrieves answers from it.**

In [ ]:
!pip install llama-index

In [ ]:
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

def index_and_query_local_file(file_name):
    """
    Indexes a local text file using LlamaIndex and allows querying it.
    """
    print(f"--- Loading documents from {file_name} ---")
    # Load the document using SimpleDirectoryReader
    documents = SimpleDirectoryReader(input_files=[file_name]).load_data()

    print("--- Creating index ---")
    # Create a vector index from the loaded documents
    index = VectorStoreIndex.from_documents(documents)

    print("--- Creating query engine ---")
    # Convert the index to a query engine
    query_engine = index.as_query_engine()

    print("--- Ready to query ---")

    # Example queries
    query1 = "What is the secret number?"
    print(f"\nQuery: {query1}")
    response1 = query_engine.query(query1)
    print(f"Response: {response1}")

    query2 = "What is LlamaIndex?"
    print(f"\nQuery: {query2}")
    response2 = query_engine.query(query2)
    print(f"Response: {response2}")

if __name__ == "__main__":
    # Ensure the local_data.txt file exists before running
    if not os.path.exists("local_data.txt"):
        print("Error: local_data.txt not found. Please create the file with content.")
    else:
        index_and_query_local_file("local_data.txt")

**Question 9: Demonstrate combining Langchain with LamaIndex to create a simple document-based Q&A chatbot.**

In [ ]:
!pip install llama-index langchain langchain-openai

In [9]:
import os

# Create the directory if it doesn't exist
os.makedirs("./legal_docs", exist_ok=True)

# Create a dummy file so LlamaIndex doesn't find an empty folder
with open("./legal_docs/sample.txt", "w") as f:
    f.write("This is a sample legal document for testing the AI chatbot.")

In [ ]:
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from langchain.agents import Tool, initialize_agent
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory

# 1. SETUP: Create the directory and a sample file to prevent the ValueError
os.makedirs("./legal_docs", exist_ok=True)
with open("./legal_docs/legal_sample.txt", "w") as f:
    f.write("The termination notice period is 30 days. Governing law: New York.")

# 2. LLAMAINDEX: Ingest and Index the documents
documents = SimpleDirectoryReader("./legal_docs").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

# 3. LANGCHAIN: Wrap the query engine as a Tool
tools = [
    Tool(
        name="Legal_Document_Search",
        func=lambda q: str(query_engine.query(q)),
        description="Search this tool for specific details within the legal documents."
    )
]

# 4. CHATBOT: Initialize LangChain Agent with Memory
llm = ChatOpenAI(model="gpt-4o", temperature=0)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent="chat-conversational-react-description",
    memory=memory,
    verbose=True
)

# 5. EXECUTE: Test the chatbot
response = agent_executor.run("What is the notice period mentioned in my documents?")
print(f"Chatbot: {response}")

**Question 10: A legal firm wants to use AI to summarize large volumes of legal
documents and retrieve relevant information quickly. Propose a solution using Langchain and LamaIndex, and explain how it would work in practice.**

For a legal firm, the best approach is a Hybrid RAG (Retrieval-Augmented Generation) system.

LlamaIndex acts as the "data expert," efficiently indexing thousands of complex legal PDFs and case files into a searchable vector database. LangChain then serves as the "orchestrator," using specialized chains (like Map-Reduce) to summarize massive documents or using AI Agents to answer specific legal queries.

In practice, when a lawyer asks a question, LlamaIndex pulls the exact relevant clauses, and LangChain processes them through an LLM to provide a cited, concise summary, ensuring every claim is backed by a specific page in the source files.